In [ ]:
RUN_NAME = "fortunate-ox-363"

In [ ]:
import logging

# Set up root logger to only print the message (no timestamp, logger name, etc.)
from loguru import logger

logging.basicConfig(format="%(message)s", level=logging.INFO)

In [ ]:
import os

os.environ["MLFLOW_TRACKING_URI"] = os.getenv(
    "MLFLOW_TRACKING_URI", "http://localhost:9090"
)

In [ ]:
from datahub_integrations.experimentation.chatbot.chatbot import (
    AI_EXPERIMENTATION_INITIALIZED,
    mlflow,
)

assert AI_EXPERIMENTATION_INITIALIZED
runs = mlflow.search_runs(
    filter_string=f"attributes.run_name  = '{RUN_NAME}'",
    output_format="list",
    order_by=["start_time DESC"],
)
assert len(runs) == 1

## Run details

In [ ]:
print(f"Analysis for Run: {RUN_NAME}")
print(f"Run ID: {runs[0].info.run_id}")
print(f"Run params: {runs[0].data.params}")
print(f"Run metrics: {runs[0].data.metrics}")
print(f"Run tags: {runs[0].data.tags}")

In [ ]:
eval_results_table = mlflow.load_table(
    "eval_results_table.json", run_ids=[runs[0].info.run_id]
)

# print(eval_results_table.columns)
print(eval_results_table.shape)

In [ ]:
from datahub_integrations.experimentation.utils import plot_hist, plot_scatter

## Failed Prompts

In [ ]:
import pandas as pd


prompts_no_response = eval_results_table[
    (eval_results_table["has_response/score"] == False)
]
print(
    "Total prompts with no response: ",
    len(prompts_no_response),
)
if len(prompts_no_response) > 0:
    pd.set_option("display.max_colwidth", None)
    display(
        prompts_no_response.groupby("error").agg(
            count=("error", "count"), prompt_ids=("prompt_id", "unique")
        )
    )
    print("Prompts with no response:")
    display(prompts_no_response[["prompt_id", "message", "tags"]])

In [ ]:
import json
from datahub_integrations.experimentation.chatbot.eval_helpers import get_token_count
from datahub_integrations.chat.chat_history import ToolResult, ChatHistory
import pathlib

TOOL_RESULT_TOKEN_LIMIT = 10000  # 200k / 20
TOOL_RESULT_CHECK_KEYS_BEYOND_TOKEN_LIMIT = 5000


def analyze_recursive(obj: dict | list, indent=0):
    if isinstance(obj, (dict, list)):
        for key, value in obj.items() if isinstance(obj, dict) else enumerate(obj):
            count = get_token_count(json.dumps(value))
            if count > TOOL_RESULT_CHECK_KEYS_BEYOND_TOKEN_LIMIT:
                logger.info(f"{' ' * indent}{key}: {count}")
                analyze_recursive(value, indent + 1)


failed_prompt_ids = prompts_no_response["prompt_id"].unique()

for prompt_id in failed_prompt_ids:
    # Each prompt's artifact is expected to be named {prompt_id}.json
    artifact_uri = f"runs:/{runs[0].info.run_id}/history/{prompt_id}.json"
    try:
        logger.info(f"=============Prompt ID: {prompt_id}===============")
        # Download the artifact file for the run
        local_path = mlflow.artifacts.download_artifacts(artifact_uri=artifact_uri)
        history = ChatHistory.load_file(pathlib.Path(local_path))
        for i, message in enumerate(history.messages):
            count = get_token_count(json.dumps(message.to_obj()))

            if isinstance(message, ToolResult) and count > TOOL_RESULT_TOKEN_LIMIT:
                logger.info(f"ToolResult with {count} tokens at index {i}")
                analyze_recursive(message.result)

    except Exception as e:
        logger.error(
            f"Could not load artifact for prompt_id {prompt_id}: {e}", exc_info=True
        )

## Trace analysis

In [ ]:
import mlflow

# Get all traces for the specific run
traces = mlflow.search_traces(
    experiment_ids=[runs[0].info.experiment_id],
    filter_string=f"attributes.run_id = '{runs[0].info.run_id}'",
    return_type="pandas",
)

print(f"Found {len(traces)} traces for run {RUN_NAME}")

In [ ]:
# First, explode the spans column to create one row per span
traces_exploded = traces.explode("spans")[["request_id", "tags", "spans"]]
traces_exploded["prompt_id"] = traces_exploded["tags"].apply(lambda x: x["prompt_id"])

print(f"Found {len(traces_exploded)} total spans for run {RUN_NAME}")


# Extract span details into separate columns using pd.json_normalize
if not traces_exploded["spans"].isna().all():
    span_details = pd.json_normalize(traces_exploded["spans"]).add_prefix("span_")

    # Combine the original trace data with span details
    traces_exploded = pd.concat(
        [
            traces_exploded.drop("spans", axis=1).reset_index(drop=True),
            span_details.reset_index(drop=True),
        ],
        axis=1,
    )

else:
    # If no spans data, just add empty span columns
    span_columns = [
        "span_name",
        "span_parent_id",
        "span_start_time",
        "span_end_time",
        "span_status_code",
        "span_status_message",
        "span_events",
        "span_context.span_id",
        "span_context.trace_id",
        "span_attributes.mlflow.traceRequestId",
        "span_attributes.mlflow.spanType",
        "span_attributes.mlflow.spanFunctionName",
        "span_attributes.mlflow.spanInputs",
        "span_attributes.mlflow.spanOutputs",
    ]
    for col in span_columns:
        traces_exploded[col] = None
    traces_exploded = traces_exploded.drop("spans", axis=1)

traces_exploded["span_execution_time_ms"] = (
    traces_exploded["span_end_time"] - traces_exploded["span_start_time"]
) / 1e6
traces_exploded["span_execution_time_sec"] = (
    traces_exploded["span_execution_time_ms"] / 1000
)
traces_exploded["span_execution_time_min"] = (
    traces_exploded["span_execution_time_sec"] / 60
)

# print(traces_exploded.columns)

### Total Time to respond

In [ ]:
run_prompt_spans = traces_exploded[traces_exploded["span_name"] == "run_prompt"]
print(f"Found {len(run_prompt_spans)} run_prompt spans for run {RUN_NAME}")

gen_next_message_spans = traces_exploded[
    traces_exploded["span_name"] == "generate_next_message"
]
print(
    f"Found {len(gen_next_message_spans)} generate_next_message spans for run {RUN_NAME}"
)
plot_hist(
    gen_next_message_spans,
    "span_execution_time_min",
    title_suffix=f"Run ID: {runs[0].info.run_id}",
)

### Run Tool Timings

In [ ]:
# traces whose span_name starts with run_tool_*
run_tool_spans = traces_exploded[
    traces_exploded["span_name"].str.startswith("run_tool_")
]

print(f"Found {len(run_tool_spans)} run_tool_ spans for run {RUN_NAME}")
plot_hist(
    run_tool_spans,
    "span_execution_time_sec",
    breakdown_col="span_attributes.tool_name",
    title_suffix=f"Run ID: {runs[0].info.run_id}",
)

##### Get Lineage Tool inputs

In [ ]:
# filter run_tool spans using get_lineage on schema field
import json


get_lineage_tool_spans = run_tool_spans[
    run_tool_spans["span_attributes.tool_name"] == '"datahub__get_lineage"'
].copy()
get_lineage_tool_spans["column_used"] = get_lineage_tool_spans[
    "span_attributes.mlflow.spanInputs"
].apply(lambda x: json.loads(x).get("column") is not None)
get_lineage_tool_spans["filters_used"] = get_lineage_tool_spans[
    "span_attributes.mlflow.spanInputs"
].apply(lambda x: json.loads(x).get("filters") is not None)
get_lineage_tool_spans["max_hops"] = get_lineage_tool_spans[
    "span_attributes.mlflow.spanInputs"
].apply(lambda x: json.loads(x).get("max_hops", 1))
get_lineage_tool_spans["upstream"] = get_lineage_tool_spans[
    "span_attributes.mlflow.spanInputs"
].apply(lambda x: json.loads(x).get("upstream", True))

print(get_lineage_tool_spans["filters_used"].value_counts())
print(get_lineage_tool_spans["column_used"].value_counts())
print(get_lineage_tool_spans["max_hops"].value_counts())
print(get_lineage_tool_spans["upstream"].value_counts())

### Tool calls per prompt

In [ ]:
prompt_wise_tool_call_count = run_tool_spans.groupby("prompt_id").agg(
    tool_call_count=("span_name", "count")
)
plot_hist(
    prompt_wise_tool_call_count,
    "tool_call_count",
    title_suffix=f"Run ID: {runs[0].info.run_id}",
)

### Failed Run Tool Calls

In [ ]:
failed_run_tool_spans = run_tool_spans[run_tool_spans["span_status_code"] != "OK"]

print(f"Found {len(failed_run_tool_spans)} failed run_tool_ spans for run {RUN_NAME}")

if len(failed_run_tool_spans) > 0:
    display(failed_run_tool_spans.groupby("span_attributes.tool_name").size())
    pd.set_option("display.max_colwidth", 300)
    display(
        failed_run_tool_spans[
            ["prompt_id", "span_name", "span_execution_time_sec", "span_status_message"]
        ].sort_values(by="span_execution_time_sec", ascending=False)
    )

    plot_hist(
        failed_run_tool_spans,
        "span_execution_time_sec",
        breakdown_col="span_attributes.tool_name",
        title_suffix=f"Failed run_tool_ spans for Run ID: {runs[0].info.run_id}",
    )

### Bedrock call timings

In [ ]:
# traces whose span_name starts with BedrockRuntime.converse*
bedrock_converse_spans = traces_exploded[
    traces_exploded["span_name"].str.startswith("BedrockRuntime.converse")
]

print(
    f"Found {len(bedrock_converse_spans)} BedrockRuntime.converse spans for run {RUN_NAME}"
)
plot_hist(
    bedrock_converse_spans,
    "span_execution_time_sec",
    title_suffix=f"Run ID: {runs[0].info.run_id}",
)

### Failed bedrock calls

In [ ]:
failed_bedrock_converse_spans = bedrock_converse_spans[
    bedrock_converse_spans["span_status_code"] != "OK"
]

print(
    f"Found {len(failed_bedrock_converse_spans)} failed BedrockRuntime.converse spans for run {RUN_NAME}"
)
if len(failed_bedrock_converse_spans) > 0:
    display(
        failed_bedrock_converse_spans[
            ["prompt_id", "span_status_code", "span_status_message"]
        ]
    )

## Token analysis

In [ ]:
# add usage column to bedrock_converse_spans to extract usage token columns
# To avoid SettingWithCopyWarning, use .loc and consider copying if needed
# If bedrock_converse_spans is a view, make a copy to ensure safe assignment
import json

bedrock_converse_spans = bedrock_converse_spans.copy()
bedrock_converse_spans.loc[:, "usage"] = bedrock_converse_spans[
    "span_attributes.mlflow.spanOutputs"
].apply(lambda x: json.loads(x).get("usage", {}) if isinstance(x, str) else {})

bedrock_converse_spans.loc[:, "input_tokens"] = bedrock_converse_spans["usage"].apply(
    lambda x: x.get("inputTokens", 0)
)
bedrock_converse_spans.loc[:, "output_tokens"] = bedrock_converse_spans["usage"].apply(
    lambda x: x.get("outputTokens", 0)
)
bedrock_converse_spans.loc[:, "total_tokens"] = bedrock_converse_spans["usage"].apply(
    lambda x: x.get("totalTokens", 0)
)
bedrock_converse_spans.loc[:, "cache_read_input_tokens"] = bedrock_converse_spans[
    "usage"
].apply(lambda x: x.get("cacheReadInputTokens", 0))
bedrock_converse_spans.loc[:, "cache_creation_input_tokens"] = bedrock_converse_spans[
    "usage"
].apply(lambda x: x.get("cacheWriteInputTokens", 0))

bedrock_converse_spans.loc[:, "total_input_tokens"] = (
    bedrock_converse_spans["input_tokens"]
    + bedrock_converse_spans["cache_read_input_tokens"]
    + bedrock_converse_spans["cache_creation_input_tokens"]
)

bedrock_converse_spans.loc[:, "total_output_tokens"] = bedrock_converse_spans[
    "output_tokens"
]

In [ ]:
bedrock_converse_spans.loc[:, "retry_attempts"] = bedrock_converse_spans[
    "span_attributes.mlflow.spanOutputs"
].apply(
    lambda x: json.loads(x).get("ResponseMetadata", {}).get("RetryAttempts", 0)
    if isinstance(x, str)
    else 0
)

plot_hist(
    bedrock_converse_spans,
    "retry_attempts",
    title_suffix=f"Run ID: {runs[0].info.run_id}",
)

In [ ]:
plot_scatter(
    bedrock_converse_spans,
    "output_tokens",
    "span_execution_time_sec",
    breakdown_col="retry_attempts",
)

plot_scatter(
    bedrock_converse_spans,
    "cache_read_input_tokens",
    "span_execution_time_sec",
    breakdown_col="retry_attempts",
)

### Total Tokens consumed per prompt

In [ ]:
plot_hist(
    bedrock_converse_spans,
    "total_tokens",
    title_suffix=f"Run ID: {runs[0].info.run_id}",
)

plot_hist(
    bedrock_converse_spans,
    "cache_read_input_tokens",
    title_suffix=f"Run ID: {runs[0].info.run_id}",
)

plot_hist(
    bedrock_converse_spans,
    "cache_creation_input_tokens",
    title_suffix=f"Run ID: {runs[0].info.run_id}",
)


plot_hist(
    bedrock_converse_spans,
    "output_tokens",
    title_suffix=f"Run ID: {runs[0].info.run_id}",
)

In [ ]:
# add column to bedrock_converse_spans to extract tool call name

bedrock_converse_spans = bedrock_converse_spans.copy()
bedrock_converse_spans.loc[:, "content"] = bedrock_converse_spans[
    "span_attributes.mlflow.spanOutputs"
].apply(
    lambda x: json.loads(x).get("output", {}).get("message", {}).get("content", [])
    if isinstance(x, str)
    else {}
)
bedrock_converse_spans.loc[:, "next_tool_call_name"] = bedrock_converse_spans[
    "content"
].apply(
    lambda x: x[-1].get("toolUse", {}).get("name", "")
    if isinstance(x, list) and len(x) > 0
    else ""
)

In [ ]:
# Excluding below cases
# - for general questions, bedrock does not make a tool call
# - for failed bedrock invocations, output will not include tool call.
bedrock_converse_spans_calling_tool = bedrock_converse_spans[
    (bedrock_converse_spans["next_tool_call_name"] != "")
    & (bedrock_converse_spans["next_tool_call_name"] != "respond_to_user")
]

# Order by start_time, group by prompt_id, and compute tool_call_output_tokens as (next row's total_tokens - current row's total_tokens - next row's output_tokens)
# The output should have one row per span (i.e., same index/rows as bedrock_converse_spans)
bedrock_converse_spans_calling_tool = bedrock_converse_spans_calling_tool.sort_values(
    ["prompt_id", "span_start_time"]
).copy()

bedrock_converse_spans_calling_tool["next_call_total_tokens"] = (
    bedrock_converse_spans_calling_tool.groupby("prompt_id")["total_tokens"].shift(-1)
)
bedrock_converse_spans_calling_tool["next_call_output_tokens"] = (
    bedrock_converse_spans_calling_tool.groupby("prompt_id")["output_tokens"].shift(-1)
)

bedrock_converse_spans_calling_tool["tool_call_output_tokens"] = (
    bedrock_converse_spans_calling_tool["next_call_total_tokens"]
    - bedrock_converse_spans_calling_tool["next_call_output_tokens"]
    - bedrock_converse_spans_calling_tool["total_tokens"]
)

### Tool wise output tokens

In [ ]:
plot_hist(
    bedrock_converse_spans_calling_tool,
    "tool_call_output_tokens",
    breakdown_col="next_tool_call_name",
    title_suffix=f"Run ID: {runs[0].info.run_id}",
)